# SIMP Topology Optimization using Gridap in Julia

UQLID Lab at NAU, 2023 

Rahaman Lab at IIT Bhubaneswar 

In the following code: 

$p$ = densities 

$pf$ = filtered densities

$pth$ = thresholded densities 

### Load required packages 

In [1]:
# FEA 
using  Gridap
using  Gridap.Geometry
using  Gridap.Fields
using  Gridap.TensorValues 
using  Gridap.CellData 

# Meshing 
using  Gmsh 
using  GridapGmsh
using GridapTopOpt

using  LinearAlgebra 

# Gardient calculation 
using  ChainRulesCore, Zygote
import ChainRulesCore: rrule 

# For MMA 
using  NLopt 

# For plotting 
#using CairoMakie, GridapMakie 

### Constant parameters 
Material properties, such as modulus of elasticity and Poisson's ratio. 
The SIMP penalty is generally assumed as 3~5. 

In [2]:
const E = 30e3        # Young's modulus
const ν = 0.3         # Poisson's ratio
const G = E/(2*(1+ν)) # Shear modulus
const α_mat = 12e-6   #(Thermal Expansion)
const κ_mat =  1.0    #(Thermal conductivity)

# const Height = 10                 # Height of the specimen
# const Heigthfact = 1.5            # Height factor for length
# const Length = Heigthfact*Height  # Length of the half specimen
# const Widthfact = 1/8             # Width factor for Width
# const Width = Height*Widthfact    # Width of the specimen
# const Loadfact = Length/40        # Load path on the half specimen
# const HeatSource_L = Length/30    # Heat source left 
# const HeatSource_H = Height/30    # Heat source right

const Height = 10                 # Height of the specimen
const Heigthfact = 2.5            # Height factor for length
const Length = Heigthfact*Height  # Length of the half specimen
const Widthfact = 1/8             # Width factor for Width
const Width = Height*Widthfact    # Width of the specimen
#const Loadfact = Length/40        # Load path on the half specimen


const l = Height/10 # Modified     # Bending length scale
const N = 0.0                     # Micropolar coupling number

const lt = l                      # Torsional length scale
const χ = 1.0                     # Polar ratio

const λₘₐₜ = 2*G*ν/(1 -2*ν)          # Different micropolar material properties
const κₘₐₜ = 2*G*N^2/(1-N^2)
const μₘₐₜ = G*(1-2*(N^2))/(1-N^2)
const γₘₐₜ = 4*G*l^2
const βₘₐₜ = 2*G*(lt^2 -2*l^2)
const αₘₐₜ = 2*G*lt^2*((1/χ) -1)

const  T0 = 0          # Initial temperature in the domain
const  TApp = 130.0     # Applied temperature at the boundary
const  Qh = 0.0        # Not used

const fApp = 4         # Mechanical load
#forVec(x) = VectorValue(0.0,-1.0,0.0) * (1 - x[3]/Width) * fApp   # Applied out-of-plane bending and torsion
forVec = VectorValue(0.0,-fApp,0.0)

VectorValue{3, Float64}(0.0, -4.0, 0.0)

In [3]:
const εelas = 1.0e-9        # void stiffness
const εtherm = 0.03         # void conductivity
const penal = 3; 

### Create the mesh or load already created mesh file using gmsh 
A GUI is also available to use gmsh 

In [4]:
# FE parameters
order = 1                                                              # Finite element order
dom = (0,Length,0,Height,0,Width)                                      # Bounding domain
nx,ny,nz = (100,40,5) 
el_size = (nx,ny,nz)#(300,100)   

(100, 40, 5)

In [5]:
model = CartesianDiscreteModel(dom, el_size) # Initialize the model
el_Δ = get_el_Δ(model)
f_Γ_D_1(x) = (x[1] ≈ 0.0) # Left side
#f_Γ_D_2(x) = (x[1] ≈ Length)      # Right side                                                                                 
f_Γ_D_3(x) = (x[2] ≈ 0.0) #&& ((x[1] < Length - Length/20)) #|| (x[1] > Length/2 + Length/40)) # Bottom face
f_Γ_D_4(x) = (x[2] ≈ Height)  # Top face
#f_Γ_D_5(x) = (Length/2 - HeatSource_L <= x[1] <= Length/2 + HeatSource_L) && (Height/2 - HeatSource_H <= x[2] <= Height/2 + HeatSource_H) #Left heat source
                                                                                
f_Γ_N(x) = (x[1] ≈ Length) && (0.0 <= x[2] <= 2*(Height/ny))# <= Length/2 + Length/40 )            # Γ_N indicator function

update_labels!(1,model,f_Γ_D_1,"Gamma_D_1")                                 

# update_labels!(3,model,f_Γ_D_2,"Gamma_D_2")

update_labels!(3,model,f_Γ_D_3,"Gamma_D_3")

update_labels!(4,model,f_Γ_D_4,"Gamma_D_4")

#update_labels!(6,model,f_Γ_D_5,"Gamma_D_5")

update_labels!(2,model,f_Γ_N,"Gamma_N")

In [6]:
path = joinpath(pwd(), "Cant_fApp_$(fApp)_TApp_$(TApp)", "N_$(N)_Lb_$(l)")
mkpath(path)
cd(path) 

In [7]:
writevtk(model, "CantBeam")

4-element Vector{Vector{String}}:
 ["CantBeam_0.vtu"]
 ["CantBeam_1.vtu"]
 ["CantBeam_2.vtu"]
 ["CantBeam_3.vtu"]

### Constitutive matrix (1 for plane stress, 2 for plane strain) 
Elasticity tensor, constitutive relation, and SIMP penalty relation are formulated here. 

In [8]:
I2 = SymTensorValue{3,Float64}(1.0, 0.0, 0.0, 1.0, 0.0, 1.0)

function σ_Tot(ε)
    σM = (λₘₐₜ*tr(ε)*one(ε) + (2*μₘₐₜ + κₘₐₜ)*(ε))
    return σM
end

E_Matrix = zeros(3,3,3)
E_Matrix[1,2,3]=1.0
E_Matrix[2,3,1]=1.0
E_Matrix[3,1,2]=1.0
E_Matrix[1,3,2]=-1.0
E_Matrix[3,2,1]=-1.0
E_Matrix[2,1,3]=-1.0
E_Tensor = ThirdOrderTensorValue(E_Matrix ...)

function σ_Temp(T)
    ε_th = -α_mat * T * I2
    σM = (λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*(ε_th))
    return σM
end

function σ_Temp0(T0)
    ε_th = -α_mat * T0 * I2
    σM = (λₘₐₜ*tr(ε_th)*one(ε_th) + (2*μₘₐₜ + κₘₐₜ)*(ε_th))
    return σM
end

function k_gradTemp(∇)
    k_grad = κ_mat * (∇)
    return k_grad
end


function ε_Skw(∇,θ)
    ∇ᵀ = transpose(∇)
    w = (0.5*(∇ᵀ - ∇)) - (E_Tensor⋅θ)
    return w
end

function σ_Cmod(ϵ_skew)
    σM = κₘₐₜ*ϵ_skew
    return σM
end

function M_mod(∇)
    M = αₘₐₜ*(∇ ⊙ I2)*I2 + βₘₐₜ*∇ + γₘₐₜ*∇
    return M
end

function Skw(u,θ)
    ∇ᵀ = transpose(∇(u))
    w = (0.5*(∇ᵀ - ∇(u)) - (E_Tensor⋅θ))
    return w
end

function Em(p)
    return (εelas + (1.0 - εelas) * p^penal)
end

function Km(p)
    return (εtherm + (1.0 - εtherm) * p^penal)
end

Km (generic function with 1 method)

### Finite Element Analysis 

In [9]:
order = 1
degree = 2*order
Ω= Triangulation(model)
dΩ= Measure(Ω,degree) 

reffe_Disp = ReferenceFE(lagrangian ,VectorValue{3,Float64},order)
V0_Disp = TestFESpace(model,reffe_Disp;conformity=:H1,dirichlet_tags=["Gamma_D_1"],
          dirichlet_masks = [(true, true,true)])

U0_Disp = TrialFESpace(V0_Disp,[VectorValue(0.0,0.0,0.0)])
uh = zero(V0_Disp)

reffe_theta = ReferenceFE(lagrangian,Float64,order)
V0_theta  = TestFESpace(model,reffe_Disp;conformity=:H1,dirichlet_tags=["Gamma_D_1"],
           dirichlet_masks = [(true, true, true)])
U0_theta = TrialFESpace(V0_theta,[VectorValue(0.0,0.0,0.0)])
θh = zero(V0_theta)

reffe_Temp = ReferenceFE(lagrangian ,Float64 ,order)
V0_Temp = TestFESpace(model ,reffe_Temp;conformity=:H1,dirichlet_tags=["Gamma_D_3","Gamma_D_4"],
          dirichlet_masks = [(true),(true)])
Th = zero(V0_Temp)
Tapp1(x) = TApp 
Tapp2(x) = T0
U0_Temp = TrialFESpace(V0_Temp ,[TApp,T0])
U0_TempAdj = TrialFESpace(V0_Temp ,[0.0 , 0.0])

V0 = MultiFieldFESpace([V0_Disp,V0_theta,V0_Temp])
U = MultiFieldFESpace([U0_Disp,U0_theta,U0_Temp])

labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels ,"Gamma_N")
Γ_Load = BoundaryTriangulation(model ;tags = LoadTagId)
dΓ_Load = Measure(Γ_Load ,degree)
n_Γ_Load = get_normal_vector(Γ_Load) 

# Γ_N_Therm = BoundaryTriangulation(model,tags=["Gamma_D_5"])
# dΓ_N_Therm = Measure(Γ_N_Therm,2*order)
# vol_D = sum(∫(1)dΩ)

p_reffe = ReferenceFE(lagrangian, Float64, 0)
Q = TestFESpace(Ω, p_reffe, vector_type = Vector{Float64})
P = Q
np = num_free_dofs(P) 

pf_reffe = ReferenceFE(lagrangian, Float64, 1)
Qf = TestFESpace(Ω, pf_reffe, vector_type = Vector{Float64})
Pf = Qf 

fem_params = (;V0_Disp, U0_Disp, Q, P, Qf, Pf, np, Ω, dΩ, n_Γ_Load, dΓ_Load) 

forVec  
function stepDisp(fem_params,pth)
    a((u,θ,T),(w,v,z)) =
        ∫( ((p->Em(p))∘pth) * (
              (ε(w) ⊙ (σ_Tot∘(ε(u))))
            + (ε(w) ⊙ (σ_Temp∘(T)))
            + ((∇(v)) ⊙ (M_mod∘(∇(θ))))
            + ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))))
            - (v ⋅ ((E_Tensor) ⋅² (σ_Cmod∘(ε_Skw∘(∇(u),θ)))))
        ) )*dΩ +
        ∫( ((p->Km(p))∘pth) * (∇(z) ⋅ (k_gradTemp∘(∇(T)))) )*dΩ

    b((w,v,z)) =
        ∫( ((p->Em(p))∘pth) * (ε(w) ⊙ (σ_Temp0(T0))) )*dΩ +
        ∫( w ⋅ forVec )*dΓ_Load

    op_Disp = AffineFEOperator(a,b,U,V0)
    uh_out,phih_out,Th_out = solve(op_Disp)
    return uh_out, phih_out, Th_out, get_matrix(op_Disp)
end

stepDisp (generic function with 1 method)

### Filtering and thresholding 
Helmholtz PDE-based filter is used here. 

(see: Lazarov, Boyan Stefanov, and Ole Sigmund. "Filters in topology optimization based on Helmholtz‐type differential equations." International Journal for Numerical Methods in Engineering 86.6 (2011): 765-781.) 

A smooth approximation of the Heaviside step function is used as the thresholding function. $\beta$ determines the strength of the approximation, and $\eta$ determines the location. Note that $\eta=0.5$ is standard in TO. 

In [10]:
elemsize = maximum(get_el_Δ(model))
r = (3*elemsize)/(2*sqrt(3)) #0.025           # Filter radius
β = 4                       # β∈[1,∞], threshold sharpness
η = 0.5                     # η∈[0,1], threshold center

a_f(r, u, v) = r^2 * (∇(v) ⋅ ∇(u))

function Filter(p0; r, fem_params)
    ph = FEFunction(fem_params.P, p0)
    op = AffineFEOperator(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ, ∫(v * ph)fem_params.dΩ
      end
    pfh = solve(op)
    return get_free_dof_values(pfh)
end

function Threshold(pfh; β, η)
    return  ((tanh(β * η) + tanh(β * (pfh - η))) / (tanh(β * η) + tanh(β * (1.0 - η)))) 

end 

NO_FIELDS = ZeroTangent() 

Dptdpf(pf, β, η) = β * (1.0 - tanh(β * (pf - η))^2) / (tanh(β * η) + tanh(β * (1.0 - η))) # Gradient of thresholding function 
DEdpf(pf, β, η)= penal * ((Threshold(pf; β, η)) ^ (penal-1)) * Dptdpf(pf, β, η) # Gradient of density^penal

#DAdpf(u, θ, T, w, v, z, pfh; β, η) =  ((p->DEdpf(p, β, η))∘pfh) * ((ε(w) ⊙ (σ_Tot∘(ε(u))) ) + ((p->DEdpf(p, β, η))∘pfh) * ε(w) ⊙ (σ_Temp∘(T)) + ((p->DEdpf(p, β, η))∘pfh) * ∇(z) ⋅ (k_gradTemp∘(∇(T))) + (((p->DEdpf(p, β, η))∘pfh) * ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) )) + (((p->DEdpf(p, β, η))∘pfh) * (∇(v))⋅ (M_mod∘(∇(θ)))) - (((p->DEdpf(p, β, η))∘pfh)* (v*((E_Matrx) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) ) - ((p->DEdpf(p, β, η))∘pfh) * ε(w) ⊙ (σ_Temp0(T0))

#DAdpf(u, θ, T, w, v, z, pfh; β, η) =  ((p->DEdpf(p, β, η))∘pfh) * ((ε(w) ⊙ (σ_Tot∘(ε(u)))) +  (ε(w) ⊙ (σ_Temp∘(T)))
#                                  + ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) ) +  ((∇(v))⋅ (M_mod∘(∇(θ)))) 
#                                  - (v*((E_Matrx) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))))) + (∇(z) ⋅ (k_gradTemp∘(∇(T)))) - (ε(w) ⊙ (σ_Temp0(T0))))

function DEdpf(pf, β, η)  ## For elastic case
    pt = Threshold(pf; β, η)
    return (1.0-εelas)*penal*(pt^(penal-1))*Dptdpf(pf,β,η)
end

function DKdpf(pf, β, η) ## For thermal case
    pt = Threshold(pf; β, η)
    return (1.0-εtherm)*penal*(pt^(penal-1))*Dptdpf(pf,β,η)
end

function DAdpf(u, θ, T, w, v, z, pfh; β, η)

    dE = (p->DEdpf(p,β,η)) ∘ pfh
    dK = (p->DKdpf(p,β,η)) ∘ pfh

    mech_terms =
        (ε(w) ⊙ (σ_Tot∘(ε(u)))) +
        (ε(w) ⊙ (σ_Temp∘(T))) +
        ((Skw(w,v)) ⊙ (σ_Cmod∘(ε_Skw∘(∇(u),θ))) ) +
        ((∇(v)) ⊙ (M_mod∘(∇(θ)))) -
        ((v ⋅ ((E_Tensor) ⋅² (σ_Cmod∘(ε_Skw∘(∇(u),θ)))))) -
        (ε(w) ⊙ (σ_Temp0(T0)))

    therm_terms =
        (∇(z) ⋅ (k_gradTemp∘(∇(T))))

    return dE*mech_terms + dK*therm_terms
end


function Dfdpf(u,θ,T,pfh; β, η)   ## For objective function
    Ah = ε(u) - α_mat*(T-T0)*I2
    σh = C2 ⊙ Ah
    DfdpfFn = ((p->DEdpf(p, β, η))∘pfh) * (σh ⊙ Ah)
    return DfdpfFn
end

Dfdpf (generic function with 1 method)

### FEA result with the entire domain filled with density 1 elements 
For checking! 

In [11]:
# Comment/Uncomment for FEA Analysis 
p0 = ones(fem_params.np)
pf_vec = Filter(p0;r, fem_params)
pfh = FEFunction(fem_params.Pf, pf_vec)
pth = (pf -> Threshold(pf; β, η)) ∘ pfh
uh, θh, Th, A_mat = stepDisp(fem_params,pth)

writevtk(Ω,"FEAresultsmbb",cellfields=["uh"=>uh,"θh"=>θh,"Th"=>Th,"p0"=>p0,"pth"=>pth]);

### Gradient calculation 
Using adjoint method 

In [12]:
C2 = isotropic_elast_tensor(3,30e3,0.3)

SymFourthOrderTensorValue{3, Float64, 36}(40384.61538461538, 0.0, 0.0, 17307.69230769231, 0.0, 17307.69230769231, 0.0, 11538.461538461537, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 11538.461538461537, 0.0, 0.0, 0.0, 17307.69230769231, 0.0, 0.0, 40384.61538461538, 0.0, 17307.69230769231, 0.0, 0.0, 0.0, 0.0, 11538.461538461537, 0.0, 17307.69230769231, 0.0, 0.0, 17307.69230769231, 0.0, 40384.61538461538)

In [13]:
function pf_p0(p0; r, fem_params)
    pf_vec = Filter(p0; r, fem_params)
    pf_vec
end

function Dgdp(dgdpf; r, fem_params)
    Af = assemble_matrix(fem_params.Pf, fem_params.Qf) do u, v
        ∫(a_f(r, u, v))fem_params.dΩ + ∫(v * u)fem_params.dΩ
    end
    wvec = Af' \ dgdpf
    wh = FEFunction(fem_params.Pf, wvec)
    l_temp(dp) = ∫(wh * dp)fem_params.dΩ
    return assemble_vector(l_temp, fem_params.P)
end 

function gf_p(p0::Vector, grad::Vector; r, β, η, fem_params)
    pf_vec = Filter(p0; r, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    pth = (pf -> Threshold(pf; β, η)) ∘ pfh
    uh, θh, Th, A_mat = stepDisp(fem_params,pth)
    Ah = ε(uh) - α_mat*(Th-T0)*I2
    σh = ((p->Em(p))∘pth) * (C2 ⊙ Ah)

    gf_values = sum(∫(σh ⊙ Ah)dΩ)  #sum(∫(uh ⋅ forVec)*dΓ_Load)
    
    dfduFn(w) = ∫(2*(σh ⊙ ε(w)))dΩ
    dfdθFn(v) = 0.0
    dfdTFn(z) = ∫(-2*α_mat*(σh ⊙ I2)*z)dΩ
    dfduVec = assemble_vector(dfduFn,fem_params.U0_Disp);
    dfdθVec = assemble_vector(dfdθFn,U0_theta);
    dfdTVec = assemble_vector(dfdTFn,U0_TempAdj);
    dfduθTVec = [-dfduVec;-dfdθVec;-dfdTVec]
    
    wvec = A_mat' \ dfduθTVec
    wUvec = wvec[1:length(dfduVec)]
    wθvec = wvec[1+length(dfduVec):1+length(dfduVec)+length(dfdθVec)]
    wTvec = wvec[1+length(dfduVec)+length(dfdθVec) : end]
    
    wUconjh = FEFunction(U0_Disp, wUvec)
    wθconjh = FEFunction(U0_theta, wθvec)
    wTconjh = FEFunction(U0_TempAdj, wTvec)
    
    wtimsdgdp(dp) = ∫((Dfdpf(uh,θh,Th,pfh; β, η))*dp)*fem_params.dΩ + ∫((DAdpf(uh, θh, Th, wUconjh, wθconjh, wTconjh, pfh; β, η)) * dp)fem_params.dΩ
    dgfdpf = assemble_vector(wtimsdgdp, fem_params.Pf)

    grad[:] = Dgdp(dgfdpf; r, fem_params)
    gradFn = FEFunction(fem_params.P, grad)

    writevtk(Ω,"FEAresultsmbbNewObj",cellfields=["uh"=>uh,"θh"=>θh,"Th"=>Th,"pth"=>pth, "pfh"=>pfh,"wUconjh"=>wUconjh,"wθconjh"=>wθconjh, "wTconjh"=>wTconjh, "gradFn"=>gradFn]);
    
     open("gvaluemma.txt","a") do io
     write(io,"$gf_values \n")
    end
    
    return gf_values
end 

gf_p (generic function with 1 method)

### Volume constraint calculation 
Volume should be estimated with filtered densities and not with thresholded densities. 

In [14]:
function  project(q,model ,dΩ,order)
    reffe = ReferenceFE(lagrangian ,Float64 ,order)
    V = FESpace(model ,reffe ,conformity =:L2)
    a(u,v) =∫(u*v)*dΩ
    b(v) =∫(v*q)*dΩ
    op = AffineFEOperator(a,b,V,V)
    qh = solve(op)
    return  qh
end

project (generic function with 1 method)

In [15]:
volfrac = 0.4

dv=get_array(∫(1)fem_params.dΩ)
getpoints = get_cell_points(Ω)

gradc = zeros(fem_params.np)
function cf_p(p0::Vector, gradc::Vector; r, β, η, fem_params)
    if length(gradc) > 0
        gradc[:] = dv
    end
    pf_vec = pf_p0(p0; r, fem_params)
    pfh = FEFunction(fem_params.Pf, pf_vec)
    pth = (pf -> Threshold(pf; β, η)) ∘ pfh
    #pro_pfh = project(pfh⊙dv,model,dΩ,order)
    pthxtr = sum(∫(pfh)*dΩ) #evaluate(pfh⊙dv,getpoints)
    return pthxtr - volfrac*sum(dv)
end; 

### MMA to optimize 
Svanberg, Krister. "The method of moving asymptotes—a new method for structural optimization." International journal for numerical methods in engineering 24.2 (1987): 359-373. 

In [16]:
open("gvaluemma$P,$TApp.txt","w") do io
end

In [17]:
function gf_p_optimize(p_init; r, β, η, TOL=1e-4, MAX_ITER, fem_params)
    ##################### Optimize #################
    opt = Opt(:LD_MMA, fem_params.np)
    opt.lower_bounds = 0.001
    opt.upper_bounds = 1
    opt.xtol_rel = TOL
    opt.maxeval = MAX_ITER
    opt.min_objective = (p0, grad) -> gf_p(p0, grad; r, β, η, fem_params)
    inequality_constraint!(opt, (p0, gradc) -> cf_p(p0, gradc; r, β, η, fem_params), 1e-8)
    (g_opt, p_opt, ret) = optimize(opt, p_init)
    @show numevals = opt.numevals # the number of function evaluations
    println("got $g_opt at $p_opt after $numevals iterations (returned $ret)")
    return g_opt, p_opt

end; 

### Run the MMA 

### Run the MMA 

In [ ]:
# p0 = rand(fem_params.np) # For randomized densities 
grad = zeros(fem_params.np)

p_opt = fill(volfrac, fem_params.np)   # Uniform Initial guess
g_opt = 0

TOL = 1e-10 # tolerance of the MMA 
MAX_ITER = 700 # Maximum iteration constraints 

g_opt, p_opt = gf_p_optimize(p_opt; r, β, η, TOL, MAX_ITER, fem_params);

###############################################
# For a list of β in the thresholding function 
#----------------------------------------------
# β_list = [4.0,8.0,16.0] 
# for bi = 1 : 3
#     β = β_list[bi]
#     g_opt, p_temp_opt = gf_p_optimize(p_opt; r, β, η, fem_params)
#     global p_opt = p_temp_opt
# end 
# @show g_opt 

### Postprocessing 
Redefine the thresholding function with a sharp increase (i.e., increase β) to get a crisp boundary 

In [ ]:
βpost=64
function Thresholdp(pfh; βpost, η)
    return ((tanh(βpost * η) + tanh(βpost * (pfh - η))) / (tanh(βpost * η) + tanh(βpost * (1.0 - η)))) 
end 

pf_vec = pf_p0(p_opt; r, fem_params)
pfh = FEFunction(fem_params.Pf, pf_vec)
pth = (pf -> Thresholdp(pf; βpost, η)) ∘ pfh; 

In [ ]:
writevtk(Ω,"Cant_result",cellfields= ["p_opt"=>p_opt,"pfh"=>pfh,"pth"=>pth]);